# PhotonTrust SDK – Quickstart Notebook

This notebook walks you through the full PhotonTrust v3 Python SDK workflow:

1. **Crosstalk prediction** – instantaneous analytic model
2. **Performance DRC report** – structured pass/fail dict
3. **Process yield estimation** – Rust-accelerated Monte Carlo with correlated variation
4. **Netlist simulation** – JAX-backed scattering matrix solver
5. **Wavelength sweep** – transmission vs wavelength chart
6. **AI Surrogate training** – automatic MLP learning after warmup runs

---
> **Setup:** activate the photonstrust venv before launching Jupyter:
> ```bash
> .venv\Scripts\Activate
> jupyter notebook notebooks/quickstart.ipynb
> ```

In [ ]:
# Install deps if needed
# !pip install matplotlib pandas

In [ ]:
import photonstrust.sdk as pt
print('PhotonTrust SDK loaded ✅')

## 1. Crosstalk Prediction

In [ ]:
xt = pt.predict_crosstalk(gap_um=1.5, length_um=100, wavelength_nm=1550)
print(f'Crosstalk: {xt:.2f} dB')

gap = pt.min_gap_for_crosstalk(target_xt_db=-30, length_um=100, wavelength_nm=1550)
print(f'Min gap for -30 dB target: {gap:.3f} µm')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gaps = np.linspace(0.5, 3.0, 40)
xts  = [pt.predict_crosstalk(gap_um=g, length_um=100, wavelength_nm=1550) for g in gaps]

plt.figure(figsize=(8, 4))
plt.plot(gaps, xts, 'b-', linewidth=2, label='Crosstalk vs Gap')
plt.axhline(-30, color='r', linestyle='--', label='-30 dB spec')
plt.xlabel('Gap (µm)')
plt.ylabel('Crosstalk (dB)')
plt.title('Parallel Waveguide Crosstalk @ 1550 nm, L=100 µm')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Full DRC Report

In [ ]:
import json

report = pt.run_drc_report(
    gap_um=1.2,
    length_um=100,
    wavelength_nm=1550,
    target_xt_db=-30,
)
print(json.dumps(report, indent=2))

## 3. Process Yield – Monte Carlo (Rust-Accelerated)

In [ ]:
metrics = [
    {
        'name': 'width_nm',
        'nominal': 500,
        'sigma': 5.0,
        'sensitivity': 1.0,
        'min_allowed': 488,
        'max_allowed': 512,
    },
    {
        'name': 'etch_depth_nm',
        'nominal': 220,
        'sigma': 3.0,
        'sensitivity': 0.8,
        'min_allowed': 213,
        'max_allowed': 227,
    },
]

result = pt.estimate_yield(metrics, mc_samples=50_000, min_required_yield=0.90)

print(f"Analytic yield : {result['analytic_yield']:.2%}")
print(f"MC yield       : {result.get('mc_yield', 'N/A')}")
print(f"Estimated yield: {result['estimated_yield']:.2%}")
print(f"Pass           : {result['pass']}")

In [ ]:
# Sweep sigma to see yield sensitivity
sigmas = np.linspace(1, 10, 20)
yields = []
for s in sigmas:
    m = [{**metrics[0], 'sigma': s}, metrics[1]]
    r = pt.estimate_yield(m, mc_samples=5_000)
    yields.append(r['estimated_yield'])

plt.figure(figsize=(8, 4))
plt.plot(sigmas, [y * 100 for y in yields], 'g-', linewidth=2)
plt.axhline(90, color='r', linestyle='--', label='90% spec')
plt.xlabel('Width sigma (nm)')
plt.ylabel('Estimated yield (%)')
plt.title('Yield sensitivity to process sigma')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4 & 5. Netlist Simulation + Wavelength Sweep

Replace `path/to/netlist.json` with your compiled netlist.

In [ ]:
import pathlib, json

# Load a sample netlist from the fixtures directory
fixture_dir = pathlib.Path('../tests/fixtures')
netlist_files = list(fixture_dir.glob('*.json'))
print('Available fixtures:', [f.name for f in netlist_files])

In [ ]:
if netlist_files:
    netlist_path = netlist_files[0]
    netlist = json.loads(netlist_path.read_text(encoding='utf-8'))
    print(f'Loaded: {netlist_path.name}')

    # Single wavelength
    result = pt.simulate_netlist(netlist, wavelength_nm=1550)
    print('\nSimulation outputs:')
    for out in result.get('outputs', []):
        print(f"  {out['node']}.{out['port']} → {out.get('power_dB', 'N/A'):.2f} dB")

In [ ]:
if netlist_files:
    import pandas as pd
    wavelengths = np.linspace(1480, 1580, 60).tolist()
    sweep = pt.simulate_netlist_sweep(netlist, wavelengths_nm=wavelengths)

    rows = []
    for res in sweep:
        wl = res.get('wavelength_nm')
        for out in res.get('outputs', []):
            rows.append({
                'wavelength_nm': wl,
                'port': f"{out['node']}.{out['port']}",
                'power_dB': out.get('power_dB'),
            })

    if rows:
        df = pd.DataFrame(rows)
        pivot = df.pivot(index='wavelength_nm', columns='port', values='power_dB')

        pivot.plot(figsize=(10, 5), title='Transmission Spectrum')
        plt.xlabel('Wavelength (nm)')
        plt.ylabel('Power (dB)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 6. AI Surrogate Auto-Training

In [ ]:
import math
from photonstrust.ai_surrogate import cached_surrogate, clear_surrogate_cache, HAS_SKLEARN
print(f'scikit-learn available: {HAS_SKLEARN}')

@cached_surrogate(domain='demo_waveguide', train_threshold=20)
def slow_physics(length_um: float, width_nm: float) -> float:
    # Simulate a slow computation (e.g. FDTD)
    import time; time.sleep(0.01)
    return -0.05 * length_um + 0.002 * (width_nm - 500) ** 2

clear_surrogate_cache()

print('Warming up surrogate (20 physics evaluations)...')
for i in range(20):
    r = slow_physics(length_um=50 + i * 5, width_nm=480 + i * 2)

print('Surrogate trained! Now running 1000 instant inferences...')
results = [slow_physics(length_um=50 + i % 20 * 5 + 0.5, width_nm=485 + i % 10 * 2) for i in range(1000)]
print(f'Got {len(results)} results instantly via MLP ✅')

---
## Summary

| Function | Time | Backend |
|---|---|---|
| `predict_crosstalk` | < 1 ms | Analytic CMT |
| `estimate_yield` (50k samples) | ~50 ms | Rust + Rayon |
| `simulate_netlist` | ~5-50 ms | JAX + XLA JIT |
| `simulate_netlist_sweep` | ~100ms for 60 pts | JAX vectorised |
| `cached_surrogate` (after warmup) | < 0.1 ms | MLP inference |

All results are deterministic, reproducible, and exportable as signed Evidence Bundles via `photonstrust bundle sign`.